In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

In [2]:
# Load the full MNIST dataset
mnist = fetch_openml('mnist_784', as_frame=False, parser='auto')

# Load data and target variables
X, y = mnist.data, mnist.target
print(f"Data shape: {X.shape}, Target shape: {y.shape}")

# Turn into NumPy arrays
X = np.array(X)
y = np.array(y)

Data shape: (70000, 784), Target shape: (70000,)


In [3]:
# Create train and test sets
split_ratio = 0.9
train_size = int(split_ratio * len(X))

# Shuffle indices and reorder X and y
np.random.seed(46)
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

# Split
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Transpose to (features, samples)
X_train = X_train.T
X_test = X_test.T

# Standardize
def standardize(X):
    mean = np.mean(X, 0)
    stdev = np.std(X, 0)
    return (X - mean) / stdev

X_train = standardize(X_train)
X_test = standardize(X_test)

# Ensure y is an integer array (sometimes fetch_openml returns strings)
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [4]:
def init_parameters():
    W1 = np.random.randn(64, 784) * 0.1
    b1 = np.random.randn(64, 1) * 0.1
    W2 = np.random.randn(10, 64) * 0.1
    b2 = np.random.randn(10, 1) * 0.1

    return W1, b1, W2, b2

def relu(Z):
    return np.maximum(0, Z)

def relu_backward(Z):
    return Z > 0

def softmax(Z):
    # axis=0 because each column is a sample
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True)) 
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def softmax_backward(A2, Y):
    return A2 - Y

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def forward(
        X,
        W1,
        b1,
        W2,
        b2
):
    Z1 = W1.dot(X) + b1
    A1 = relu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

def backward(
        X,
        Y,
        Z1,
        A1, 
        Z2,
        A2,
        W2
):
    m = Y.size

    # dL/dZ2 = A2 - Y from softmax and cross-entropy loss
    dZ2 = softmax_backward(A2, one_hot(Y))

    # Z2 = W2 * A1 + b2
    # dL/dW2 = dL/dZ2 * dZ2/dW2 = dL/dZ2 * A1
    # dL/db2 = dL/dZ2 * dZ2/db2 = dL/dZ2 * 1 (sum over samples)
    dW2 = np.dot(dZ2, A1.T) * (1 / m)
    db2 = np.sum(dZ2, axis=1, keepdims=True) * (1 / m)

    # Pass the gradient back through the ReLU activation to the previous layer
    # dL/dZ1 = dL/dA1 * dA1/dZ1
    # dL/dA1 = dL/dZ2 * dZ2/dA1 = dZ2 * W2 (Z2 = W2 * A1 + b2)
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_backward(Z1)

    # Z1 = W1 * X + b1
    # dL/dW1 = dL/dZ1 * dZ1/dW1 = dL/dZ1 * X
    # dL/db1 = dL/dZ1 * dZ1/db1 = dL/dZ1 * 1 (sum over samples)
    dW1 = np.dot(dZ1, X.T) * (1 / m)
    db1 = np.sum(dZ1, axis=1, keepdims=True) * (1 / m)

    return dW1, db1, dW2, db2

def update_params(
        W1,
        b1,
        W2, 
        b2,
        dW1, 
        db1, 
        dW2,
        db2,
        stepsize
):
    W1 = W1 - stepsize * dW1
    b1 = b1 - stepsize * db1

    W2 = W2 - stepsize * dW2
    b2 = b2 - stepsize * db2

    return W1, b1, W2, b2

In [5]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size

def gradient_descent(iterations, stepsize, X, Y):
    W1, b1, W2, b2 = init_parameters()

    for i in range(iterations):
        Z1, A1, Z2, A2 = forward(X, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward(X, Y, Z1, A1, Z2, A2, W2)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, stepsize)

        if (i % 10 == 0):
            print(f'--- Iteration : {i}')
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))

    return W1, b1, W2, b2

In [6]:
W1, b1, W2, b2 = gradient_descent(500, 3e-2, X_train, y_train)

--- Iteration : 0
0.11584126984126984
--- Iteration : 10
0.4281904761904762
--- Iteration : 20
0.6007301587301588
--- Iteration : 30
0.6855238095238095
--- Iteration : 40
0.7339682539682539
--- Iteration : 50
0.7632063492063492
--- Iteration : 60
0.7834444444444445
--- Iteration : 70
0.7982539682539682
--- Iteration : 80
0.8108888888888889
--- Iteration : 90
0.8211587301587302
--- Iteration : 100
0.8292063492063492
--- Iteration : 110
0.8364920634920635
--- Iteration : 120
0.8428412698412698
--- Iteration : 130
0.8477301587301588
--- Iteration : 140
0.8521746031746031
--- Iteration : 150
0.8561428571428571
--- Iteration : 160
0.8600476190476191
--- Iteration : 170
0.8632063492063492
--- Iteration : 180
0.8659365079365079
--- Iteration : 190
0.8685079365079366
--- Iteration : 200
0.8708571428571429
--- Iteration : 210
0.8734126984126984
--- Iteration : 220
0.8758571428571429
--- Iteration : 230
0.8782222222222222
--- Iteration : 240
0.8801269841269841
--- Iteration : 250
0.8819365079365